# Toolsmith Evening 1: raw llama.cpp serving

Run this notebook in a Colab L4 runtime. The next code cell clones [bare-metal-agent](https://github.com/shreyasnarahari/bare-metal-agent) into `/content/bare-metal-agent` when it is not already on the VM.

Put `HF_TOKEN` and `LOCAL_KEY` in `MyDrive/toolsmith/.env`. When that file is absent, the Colab website supplies the same names from Colab Secrets. `LOCAL_KEY` protects the local llama.cpp server. Record the Colab units shown by the resource panel before and after the session; the notebook cannot read that value automatically.

In [1]:
import json
import os
import platform
import shlex
import shutil
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

gpu_result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
    check=True,
    capture_output=True,
    text=True,
)
gpu_info = gpu_result.stdout.strip()
print(gpu_info)
print(f'Python {platform.python_version()}')

NVIDIA L4, 23034 MiB, 580.82.07
Python 3.13.16


In [2]:
from google.colab import drive
import subprocess

REPO_URL = 'https://github.com/shreyasnarahari/bare-metal-agent'
PROJECT_DIR = Path('/content/bare-metal-agent')

if (PROJECT_DIR / '.git').is_dir():
    subprocess.run(
        ['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'],
        check=True,
    )
elif PROJECT_DIR.exists():
    raise FileNotFoundError(
        f'{PROJECT_DIR} exists but is not a bare-metal-agent git checkout.'
    )
else:
    subprocess.run(
        ['git', 'clone', REPO_URL, str(PROJECT_DIR)],
        check=True,
    )

if not (PROJECT_DIR / 'pyproject.toml').is_file():
    raise FileNotFoundError(
        f'Expected {PROJECT_DIR / "pyproject.toml"} after cloning {REPO_URL}.'
    )

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/toolsmith')
DRIVE_BUILD_CACHE = DRIVE_ROOT / 'builds'
DRIVE_MODEL_CACHE = DRIVE_ROOT / 'models'
DRIVE_RECORDS = DRIVE_ROOT / 'records'
for directory in (DRIVE_BUILD_CACHE, DRIVE_MODEL_CACHE, DRIVE_RECORDS):
    directory.mkdir(parents=True, exist_ok=True)

def read_env_file(path):
    values = {}
    if not path.is_file():
        return values
    for raw_line in path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue
        if line.startswith('export '):
            line = line[len('export '):].strip()
        key, separator, value = line.partition('=')
        if not separator:
            continue
        value = value.strip()
        if len(value) >= 2 and value[0] == value[-1] and value[0] in '"\'':
            value = value[1:-1]
        values[key.strip()] = value
    return values

def load_secret(name, env_values):
    value = env_values.get(name, '').strip()
    if value:
        return value, env_path
    from google.colab import userdata
    return userdata.get(name), 'Colab Secrets'

env_path = next(
    (
        candidate
        for candidate in (DRIVE_ROOT / '.env', DRIVE_ROOT / '.env.txt')
        if candidate.is_file()
    ),
    DRIVE_ROOT / '.env',
)
env_values = read_env_file(env_path)
HF_TOKEN, hf_source = load_secret('HF_TOKEN', env_values)
LOCAL_KEY, local_source = load_secret('LOCAL_KEY', env_values)
if not HF_TOKEN or not LOCAL_KEY:
    present = sorted(p.name for p in DRIVE_ROOT.iterdir())
    raise ValueError(
        f'HF_TOKEN and LOCAL_KEY must be set in {env_path} or in Colab Secrets. '
        f'Entries in {DRIVE_ROOT}: {present}'
    )
print(
    f'HF_TOKEN from {hf_source} ({len(HF_TOKEN)} characters); '
    f'LOCAL_KEY from {local_source} ({len(LOCAL_KEY)} characters)'
)

Mounted at /content/drive
HF_TOKEN from /content/drive/MyDrive/toolsmith/.env.txt (37 characters); LOCAL_KEY from /content/drive/MyDrive/toolsmith/.env.txt (64 characters)


In [3]:
get_ipython().run_line_magic(
    'pip',
    f"install -q -e {shlex.quote(str(PROJECT_DIR) + '[colab]')}",
)

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for toolsmith (pyproject.toml) ... done


## Build or restore llama.cpp

The revision is pinned so that later parser fixtures remain reproducible. The build is cached on Drive, then copied to Colab's local disk before use. Change the revision deliberately and record any resulting protocol differences.

In [4]:
LLAMA_CPP_REF = 'b5401'
LOCAL_SOURCE = Path('/content/llama.cpp-source')
LOCAL_BUILD = Path('/content/llama.cpp-build')
CACHED_BUILD = DRIVE_BUILD_CACHE / f'llama.cpp-{LLAMA_CPP_REF}-cuda'
SERVER_BIN = LOCAL_BUILD / 'bin' / 'llama-server'

if not SERVER_BIN.is_file() and (CACHED_BUILD / 'bin' / 'llama-server').is_file():
    shutil.copytree(CACHED_BUILD, LOCAL_BUILD, dirs_exist_ok=True)

if not SERVER_BIN.is_file():
    if not (LOCAL_SOURCE / '.git').exists():
        subprocess.run(
            ['git', 'clone', '--filter=blob:none', 'https://github.com/ggml-org/llama.cpp.git', str(LOCAL_SOURCE)],
            check=True,
        )
    subprocess.run(
        ['git', '-C', str(LOCAL_SOURCE), 'fetch', '--depth', '1', 'origin', 'tag', LLAMA_CPP_REF],
        check=True,
    )
    subprocess.run(
        ['git', '-C', str(LOCAL_SOURCE), 'checkout', '--detach', LLAMA_CPP_REF],
        check=True,
    )
    subprocess.run(
        [
            'cmake', '-S', str(LOCAL_SOURCE), '-B', str(LOCAL_BUILD),
            '-DGGML_CUDA=ON', '-DCMAKE_BUILD_TYPE=Release', '-DLLAMA_CURL=ON',
        ],
        check=True,
    )
    subprocess.run(
        ['cmake', '--build', str(LOCAL_BUILD), '--target', 'llama-server', '-j', '2'],
        check=True,
    )
    shutil.copytree(LOCAL_BUILD, CACHED_BUILD, dirs_exist_ok=True)

if not SERVER_BIN.is_file():
    raise FileNotFoundError(f'llama-server was not built at {SERVER_BIN}')

SERVER_BIN.chmod(SERVER_BIN.stat().st_mode | 0o755)

library_path = str(LOCAL_BUILD / 'lib')
os.environ['LD_LIBRARY_PATH'] = library_path + os.pathsep + os.environ.get('LD_LIBRARY_PATH', '')
print(f'Using llama.cpp {LLAMA_CPP_REF}: {SERVER_BIN}')

Using llama.cpp b5401: /content/llama.cpp-build/bin/llama-server


In [5]:
from pathlib import Path
p = Path('/content/drive/MyDrive/toolsmith/builds/llama.cpp-b5401-cuda/bin/llama-server')
print('is_symlink', p.is_symlink())
print('is_file', p.is_file())
try:
    print('lstat', oct(p.lstat().st_mode), p.lstat().st_size)
except OSError as error:
    print('lstat failed', error)
if p.is_symlink():
    print('target', p.readlink())

is_symlink False
is_file True
lstat 0o100600 5208384


In [6]:
from huggingface_hub import hf_hub_download

MODEL_REPO = 'Qwen/Qwen3-8B-GGUF'
MODEL_FILE = 'Qwen3-8B-Q4_K_M.gguf'
LOCAL_MODEL = Path('/content') / MODEL_FILE

cached_model = Path(
    hf_hub_download(
        repo_id=MODEL_REPO,
        filename=MODEL_FILE,
        cache_dir=str(DRIVE_MODEL_CACHE),
        token=HF_TOKEN,
    )
)
if not LOCAL_MODEL.exists() or LOCAL_MODEL.stat().st_size != cached_model.stat().st_size:
    shutil.copy2(cached_model, LOCAL_MODEL)

print(f'Local model: {LOCAL_MODEL} ({LOCAL_MODEL.stat().st_size:,} bytes)')

Local model: /content/Qwen3-8B-Q4_K_M.gguf (79 bytes)


In [8]:
import sys

src_dir = PROJECT_DIR / 'src'
if not (src_dir / 'toolsmith' / '__init__.py').is_file():
    raise FileNotFoundError(
        f'{src_dir / "toolsmith"} is not in the Colab checkout. '
        f'Run git -C {PROJECT_DIR} pull --ff-only, then rerun the install cell.'
    )
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from toolsmith.backends import (
    CompletionTruncatedError,
    LlamaCppClient,
    LlamaCppServer,
)
from toolsmith.recording import JsonlRecorder

SERVER_LOG_PATH = Path('/content/llama-server.log')
import socket

def release_port(port: int) -> None:
    previous = globals().get('server')
    if previous is not None and previous.is_running:
        previous.stop(timeout=10)
    probe = socket.socket()
    try:
        probe.bind(('127.0.0.1', port))
    except OSError:
        subprocess.run(
            ['fuser', '-k', f'{port}/tcp'],
            check=False,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        time.sleep(1)
    finally:
        probe.close()

release_port(8080)
SERVER_BIN.chmod(SERVER_BIN.stat().st_mode | 0o755)
server = LlamaCppServer(
    executable=SERVER_BIN,
    model=LOCAL_MODEL,
    api_key=LOCAL_KEY,
    log_path=SERVER_LOG_PATH,
    port=8080,
    context_size=16384,
    gpu_layers=99,
)
server.start()
print(f'Server PID: {server.process.pid}; log: {SERVER_LOG_PATH}')

ImportError: cannot import name 'LlamaCppServer' from 'toolsmith.backends' (/content/bare-metal-agent/src/toolsmith/backends/__init__.py)

In [ ]:
client = LlamaCppClient(server.base_url, LOCAL_KEY, timeout=10.0)
health = server.wait_until_healthy(client, timeout=600, poll_interval=2)
print(json.dumps(health, indent=2))

## Raw completion smoke test

This request intentionally uses `/completion`, not llama.cpp's chat endpoint. Toolsmith will own prompt rendering and tool-call parsing in later evenings.

In [ ]:
prompt = 'Name one primary color. Respond with one word only.\nAnswer:'
SEED = 0
started = time.perf_counter()
try:
    response = client.complete(
        prompt,
        max_tokens=128,
        temperature=0.0,
        seed=SEED,
    )
    outcome = 'ok'
except CompletionTruncatedError as error:
    response = error.response
    outcome = 'truncated'
elapsed_seconds = time.perf_counter() - started

record = {
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'llama_cpp_ref': LLAMA_CPP_REF,
    'model_repo': MODEL_REPO,
    'model_file': MODEL_FILE,
    'gpu': gpu_info,
    'python_version': platform.python_version(),
    'server': server.settings(),
    'request': {
        'prompt': prompt,
        'max_tokens': 128,
        'temperature': 0.0,
        'seed': SEED,
        'stream': False,
    },
    'response': response,
    'outcome': outcome,
    'elapsed_seconds': elapsed_seconds,
}
record_path = DRIVE_RECORDS / 'evening-01-smoke.jsonl'
JsonlRecorder(record_path, secrets=(HF_TOKEN, LOCAL_KEY)).append(record)

print(json.dumps(response, indent=2))
print(f'Elapsed: {elapsed_seconds:.2f}s; record: {record_path}')
if outcome != 'ok':
    raise RuntimeError('The smoke test was truncated; inspect the recorded response before continuing.')

In [ ]:
if 'client' in globals():
    client.close()
if 'server' in globals():
    server.stop(timeout=20)

print('Server stopped. Record Colab units used, then delete or disconnect the runtime.')